In [0]:
from pyspark.sql.functions import *

In [0]:
# Reading the CSV file into a Spark DataFrame
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/celebal/w6_dataset/dataset.csv")

print("Dataset loaded successfully.")

Dataset loaded successfully.


Viewing Dataset and Schema

In [0]:
# Display first few records
df.show(5)
# Check the data types of all columns
df.printSchema()

+--------+-----------+------------+-----------+--------+----------+--------+------+------------+----------+------------+------------+
|order_id|customer_id|product_name|   category|quantity|unit_price|discount|region|order_status|order_date|payment_mode|sales_person|
+--------+-----------+------------+-----------+--------+----------+--------+------+------------+----------+------------+------------+
|       1|   CUST1001|       Mouse|Electronics|       1|      1000|     5.0| South|   Completed|2024-05-31|        Cash|        Riya|
|       2|   CUST1002|         Pen| Stationery|       1|       150|     0.0| South|   Completed|2024-06-15|        Cash|        Neha|
|       3|   CUST1003|    Keyboard|Electronics|       7|       750|    15.0|  East|   Completed|2024-03-28|        Card|       Rahul|
|       4|   CUST1004|    Notebook| Stationery|       3|       750|    10.0| North|   Completed|2024-04-01| Net Banking|        Riya|
|       5|   CUST1005|     Printer|Electronics|      10|      

In [0]:
print("Total Records:", df.count())

Total Records: 152


Selecting and Filtering Data

In [0]:
# Selecting only the required columns for further processing
selected_df = df.select("order_id", "product_name", "category", "quantity", "unit_price", "region")
selected_df.show(5)

+--------+------------+-----------+--------+----------+------+
|order_id|product_name|   category|quantity|unit_price|region|
+--------+------------+-----------+--------+----------+------+
|       1|       Mouse|Electronics|       1|      1000| South|
|       2|         Pen| Stationery|       1|       150| South|
|       3|    Keyboard|Electronics|       7|       750|  East|
|       4|    Notebook| Stationery|       3|       750| North|
|       5|     Printer|Electronics|      10|      1000|  West|
+--------+------------+-----------+--------+----------+------+
only showing top 5 rows


In [0]:
# Filtering records
filtered_df = selected_df.filter((col("region") == "North") & (col("quantity") >= 5))
filtered_df.show()

+--------+------------+-----------+--------+----------+------+
|order_id|product_name|   category|quantity|unit_price|region|
+--------+------------+-----------+--------+----------+------+
|       7|        Desk|  Furniture|       5|       250| North|
|      21|     Printer|Electronics|       8|      7500| North|
|      27|       Chair|  Furniture|       5|      5000| North|
|      47|        Desk|  Furniture|       7|      1500| North|
|      50|        Desk|  Furniture|       5|       150| North|
|      51|       Mouse|Electronics|       6|     12000| North|
|      55|        Desk|  Furniture|       9|      5000| North|
|      62|        Desk|  Furniture|       5|      1500| North|
|      64|        Desk|  Furniture|       7|      2500| North|
|      67|     Cabinet|  Furniture|       9|       150| North|
|      69|         Pen| Stationery|       5|      2500| North|
|      90|    Notebook| Stationery|      10|      2500| North|
|      95|     Cabinet|  Furniture|      10|      1000|

Modifying the DataFrame

In [0]:
# Renameing the price column
renamed_df = filtered_df.withColumnRenamed("unit_price", "price")
renamed_df.show(5)

+--------+------------+-----------+--------+-----+------+
|order_id|product_name|   category|quantity|price|region|
+--------+------------+-----------+--------+-----+------+
|       7|        Desk|  Furniture|       5|  250| North|
|      21|     Printer|Electronics|       8| 7500| North|
|      27|       Chair|  Furniture|       5| 5000| North|
|      47|        Desk|  Furniture|       7| 1500| North|
|      50|        Desk|  Furniture|       5|  150| North|
+--------+------------+-----------+--------+-----+------+
only showing top 5 rows


In [0]:
# Converting the price column to double data type
cast_df = renamed_df.withColumn("price",col("price").cast("double"))
cast_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- region: string (nullable = true)



In [0]:
# Creating total amount column
updated_df = cast_df.withColumn("total_amount", col("quantity") * col("price"))
updated_df.show(5)

+--------+------------+-----------+--------+------+------+------------+
|order_id|product_name|   category|quantity| price|region|total_amount|
+--------+------------+-----------+--------+------+------+------------+
|       7|        Desk|  Furniture|       5| 250.0| North|      1250.0|
|      21|     Printer|Electronics|       8|7500.0| North|     60000.0|
|      27|       Chair|  Furniture|       5|5000.0| North|     25000.0|
|      47|        Desk|  Furniture|       7|1500.0| North|     10500.0|
|      50|        Desk|  Furniture|       5| 150.0| North|       750.0|
+--------+------------+-----------+--------+------+------+------------+
only showing top 5 rows


In [0]:
# Handling null values
clean_df = updated_df.na.fill({"region": "Unknown"})
clean_df.show(5)

+--------+------------+-----------+--------+------+------+------------+
|order_id|product_name|   category|quantity| price|region|total_amount|
+--------+------------+-----------+--------+------+------+------------+
|       7|        Desk|  Furniture|       5| 250.0| North|      1250.0|
|      21|     Printer|Electronics|       8|7500.0| North|     60000.0|
|      27|       Chair|  Furniture|       5|5000.0| North|     25000.0|
|      47|        Desk|  Furniture|       7|1500.0| North|     10500.0|
|      50|        Desk|  Furniture|       5| 150.0| North|       750.0|
+--------+------------+-----------+--------+------+------+------------+
only showing top 5 rows


In [0]:
# Removing duplicate records
duplicate_count = clean_df.count()
no_duplicate_df = clean_df.dropDuplicates()
print("Records before removing duplicates:", duplicate_count)
print("Records after removing duplicates:", no_duplicate_df.count())

Records before removing duplicates: 27
Records after removing duplicates: 26


In [0]:
# Grouping data by category
from pyspark.sql.functions import count,sum,avg,min,max
group_df=no_duplicate_df.groupBy("category").agg(count("*").alias("total_orders"),sum("total_amount").alias("total_sales"),avg("price").alias("average_price"),min("price").alias("minimum_price"),max("price").alias("maximum_price"))
group_df.show()

+-----------+------------+-----------+------------------+-------------+-------------+
|   category|total_orders|total_sales|     average_price|minimum_price|maximum_price|
+-----------+------------+-----------+------------------+-------------+-------------+
| Stationery|           7|    58500.0|            1200.0|        150.0|       2500.0|
|  Furniture|           9|   118850.0|1894.4444444444443|        150.0|       5000.0|
|Electronics|          10|   401500.0|            5175.0|        500.0|      12000.0|
+-----------+------------+-----------+------------------+-------------+-------------+



Understanding Lazy Evaluation and DAG

In [0]:
# Creating lazy transformation
lazy_df=no_duplicate_df.filter(col("price")>1000)
print("No action has been performed yet.")

No action has been performed yet.


In [0]:
# Viewing execution plan
lazy_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonGroupingAgg(keys=[order_id#15190, product_name#15192, category#15193, quantity#15194, price#15206, region#15209, total_amount#15208], functions=[])
         +- PhotonShuffleExchangeSource
            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#13123]
               +- PhotonShuffleExchangeSink hashpartitioning(order_id#15190, product_name#15192, category#15193, quantity#15194, price#15206, region#15209, total_amount#15208, 16)
                  +- PhotonGroupingAgg(keys=[order_id#15190, product_name#15192, category#15193, quantity#15194, knownfloatingpointnormalized(normalizenanandzero(price#15206)) AS price#15206, region#15209, knownfloatingpointnormalized(normalizenanandzero(total_amount#15208)) AS total_amount#15208], functions=[])
                     +- PhotonProject [order_id#15190, product_name#15192, category#15193, quantity#15194, p

In [0]:
# running action
lazy_df.show(5)

+--------+------------+-----------+--------+------+------+------------+
|order_id|product_name|   category|quantity| price|region|total_amount|
+--------+------------+-----------+--------+------+------+------------+
|      47|        Desk|  Furniture|       7|1500.0| North|     10500.0|
|      21|     Printer|Electronics|       8|7500.0| North|     60000.0|
|      55|        Desk|  Furniture|       9|5000.0| North|     45000.0|
|      62|        Desk|  Furniture|       5|1500.0| North|      7500.0|
|      69|         Pen| Stationery|       5|2500.0| North|     12500.0|
+--------+------------+-----------+--------+------+------+------------+
only showing top 5 rows


In [0]:
# Saving data in CSV format
lazy_df.write.mode("overwrite").option("header",True).csv("/Volumes/workspace/celebal/w6_dataset/csv_output")
print("CSV file saved successfully.")

CSV file saved successfully.


In [0]:
# Saving data in Parquet format
lazy_df.write.mode("overwrite").parquet("/Volumes/workspace/celebal/w6_dataset/parquet_output")
print("Parquet file saved successfully.")

Parquet file saved successfully.


Working with Parquet Files

In [0]:
# Reading Parquet file
parquet_df=spark.read.parquet("/Volumes/workspace/celebal/w6_dataset/parquet_output")
parquet_df.show(5)
parquet_df.printSchema()

+--------+------------+-----------+--------+------+------+------------+
|order_id|product_name|   category|quantity| price|region|total_amount|
+--------+------------+-----------+--------+------+------+------------+
|      47|        Desk|  Furniture|       7|1500.0| North|     10500.0|
|      21|     Printer|Electronics|       8|7500.0| North|     60000.0|
|      55|        Desk|  Furniture|       9|5000.0| North|     45000.0|
|      62|        Desk|  Furniture|       5|1500.0| North|      7500.0|
|      69|         Pen| Stationery|       5|2500.0| North|     12500.0|
+--------+------------+-----------+--------+------+------+------------+
only showing top 5 rows
root
 |-- order_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- region: string (nullable = true)
 |-- total_amount: double (nullable = true)



In [0]:
# Filtering Parquet data
parquet_filter=parquet_df.filter(col("price")>1000)
parquet_filter.show(5)

+--------+------------+-----------+--------+------+------+------------+
|order_id|product_name|   category|quantity| price|region|total_amount|
+--------+------------+-----------+--------+------+------+------------+
|      47|        Desk|  Furniture|       7|1500.0| North|     10500.0|
|      21|     Printer|Electronics|       8|7500.0| North|     60000.0|
|      55|        Desk|  Furniture|       9|5000.0| North|     45000.0|
|      62|        Desk|  Furniture|       5|1500.0| North|      7500.0|
|      69|         Pen| Stationery|       5|2500.0| North|     12500.0|
+--------+------------+-----------+--------+------+------+------------+
only showing top 5 rows


In [0]:
# Comparing CSV and Parquet files

csv_df=spark.read.csv("/Volumes/workspace/celebal/w6_dataset/dataset.csv",header=True,inferSchema=True)
print("CSV Schema")
csv_df.printSchema()

print("Parquet Schema")
parquet_df.printSchema()

CSV Schema
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount: double (nullable = true)
 |-- region: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- sales_person: string (nullable = true)

Parquet Schema
root
 |-- order_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- region: string (nullable = true)
 |-- total_amount: double (nullable = true)



In [0]:
# Selecting required columns
parquet_df.select("product_name","category","price","total_amount").show(5)

+------------+-----------+------+------------+
|product_name|   category| price|total_amount|
+------------+-----------+------+------------+
|        Desk|  Furniture|1500.0|     10500.0|
|     Printer|Electronics|7500.0|     60000.0|
|        Desk|  Furniture|5000.0|     45000.0|
|        Desk|  Furniture|1500.0|      7500.0|
|         Pen| Stationery|2500.0|     12500.0|
+------------+-----------+------+------------+
only showing top 5 rows


Building the Data Processing Pipeline

In [0]:
# Building Spark pipeline

pipeline_df=spark.read.csv("/Volumes/workspace/celebal/w6_dataset/dataset.csv",header=True,inferSchema=True)
pipeline_df=pipeline_df.dropDuplicates()
pipeline_df=pipeline_df.na.fill({"region":"Unknown"})
pipeline_df=pipeline_df.filter(col("order_status")=="Completed")
pipeline_df=pipeline_df.withColumn("total_amount",col("quantity")*col("unit_price"))
pipeline_df.write.mode("overwrite").parquet("/Volumes/workspace/celebal/w6_dataset/final_pipeline")
pipeline_df.show(10)
print("Pipeline executed successfully")

+--------+-----------+------------+-----------+--------+----------+--------+------+------------+----------+------------+------------+------------+
|order_id|customer_id|product_name|   category|quantity|unit_price|discount|region|order_status|order_date|payment_mode|sales_person|total_amount|
+--------+-----------+------------+-----------+--------+----------+--------+------+------------+----------+------------+------------+------------+
|      72|   CUST1072|     Cabinet|  Furniture|      10|       250|     5.0|  East|   Completed|2024-05-27|        Cash|       Vikas|        2500|
|     116|   CUST1116|       Chair|  Furniture|       4|      7500|    15.0| South|   Completed|2024-06-05|        Card|        Riya|       30000|
|     119|   CUST1119|    Notebook| Stationery|       7|      1000|    15.0|  East|   Completed|2024-01-28| Net Banking|       Priya|        7000|
|     133|   CUST1133|         Pen| Stationery|       5|       750|     0.0|  West|   Completed|2024-01-13|        Car

In [0]:
# Displaying final output
pipeline_df.show(10)

+--------+-----------+------------+-----------+--------+----------+--------+------+------------+----------+------------+------------+------------+
|order_id|customer_id|product_name|   category|quantity|unit_price|discount|region|order_status|order_date|payment_mode|sales_person|total_amount|
+--------+-----------+------------+-----------+--------+----------+--------+------+------------+----------+------------+------------+------------+
|      72|   CUST1072|     Cabinet|  Furniture|      10|       250|     5.0|  East|   Completed|2024-05-27|        Cash|       Vikas|        2500|
|     116|   CUST1116|       Chair|  Furniture|       4|      7500|    15.0| South|   Completed|2024-06-05|        Card|        Riya|       30000|
|     119|   CUST1119|    Notebook| Stationery|       7|      1000|    15.0|  East|   Completed|2024-01-28| Net Banking|       Priya|        7000|
|     133|   CUST1133|         Pen| Stationery|       5|       750|     0.0|  West|   Completed|2024-01-13|        Car

## Observations

1. Spark DataFrames made data processing simple and efficient.
2. Lazy Evaluation executed transformations only after an action was called.
3. The execution plan was viewed using explain().
4. The dataset was successfully saved in both CSV and Parquet formats.
5. Parquet is more efficient for Spark processing because it is a columnar file format.
6. The complete data pipeline was successfully implemented using Spark DataFrames.